## 1. Load and Combine Training Data

Load the PostgreSQL and Stripe datasets, then combine them into a single DataFrame for preprocessing.

In [222]:
import pandas as pd

In [223]:
# Load the datasets (Stripe API & Postgres)

postgres_df = pd.read_csv('dataset/postgres_data.csv')
stripe_df = pd.read_csv('dataset/stripe_data.csv')
stripe_df_1 = pd.read_csv('dataset/stripe_data_1.csv')

In [224]:
# Add new columns to the postrgres dataset to match the Stripe dataset
postgres_df["transaction_timestamp"] = pd.NA
postgres_df["currency"] = pd.NA

In [225]:
# Concatinate the three datasets into one dataframe
df = pd.concat([postgres_df, stripe_df, stripe_df_1], ignore_index=True)

## 2. Data Preprocessing

Clean, standardize, and prepare the combined dataset for anomaly detection.

In [226]:
# Drop the columns that are not needed for anomaly detection

columns_to_drop = ["transaction_id", "customer_id", "card_id", "merchant_id", "transaction_timestamp", "currency", "postal_code", "country", "merchant_city"]
df = df.drop(columns=columns_to_drop)

In [227]:
# check missing values
df.isnull().sum()

amount                         0
card_usage_method              0
merchant_state             23471
merchant_category_code         0
transaction_error         198400
age                            0
gender                         0
annual_income                  0
credit_score                   0
number_of_cards                0
card_brand                     0
card_type                      0
has_chip                       0
credit_limit                   0
dtype: int64

In [228]:
# Fill missing merchant states
df["merchant_state"] = df["merchant_state"].fillna("Unknown")

In [229]:
# Fill missing transaction errors with "Success" (An empty value already indicates a successful transaction.)
df["transaction_error"] = (
    df["transaction_error"].fillna("Success")
)

In [230]:
# Normalize some Categorical Values (card_usage_method, card_brand and card_type)

df["card_usage_method"] = df["card_usage_method"].replace({
    "Swipe Transaction": "Swipe",
    "Chip Transaction": "Chip",
    "Online Transaction": "Online"
})


df["card_brand"] = df["card_brand"].replace({
    "visa": "Visa"
})


df["card_type"] = df["card_type"].replace({
    "credit": "Credit"
})

In [231]:
print(df["card_usage_method"].value_counts())
print(df["card_brand"].value_counts())
print(df["card_type"].value_counts())

card_usage_method
Swipe          105186
Chip            72187
Online          23764
Contactless       363
Name: count, dtype: int64
card_brand
Mastercard    107367
Visa           76287
Amex           12780
Discover        5066
Name: count, dtype: int64
card_type
Debit              124360
Credit              63341
Debit (Prepaid)     13799
Name: count, dtype: int64


In [232]:
# Encode binary categorical features

df["gender"] = df["gender"].map({
    "Male": 0,
    "Female": 1
})

df["has_chip"] = df["has_chip"].astype(int)

df["transaction_error"] = (
    df["transaction_error"]
    .apply(lambda x: 0 if x == "Success" else 1)
)

In [233]:
# Standardize US state codes
state_mapping = {
    "AA": "Armed Forces Americas",
    "AK": "Alaska",
    "AL": "Alabama",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DC": "District of Columbia",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "IA": "Iowa",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "MA": "Massachusetts",
    "MD": "Maryland",
    "ME": "Maine",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MO": "Missouri",
    "MS": "Mississippi",
    "MT": "Montana",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "NE": "Nebraska",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NV": "Nevada",
    "NY": "New York",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VA": "Virginia",
    "VT": "Vermont",
    "WA": "Washington",
    "WI": "Wisconsin",
    "WV": "West Virginia",
    "WY": "Wyoming"
}

# Apply the mapping and keep other values unchanged
df["merchant_state_normalized"] = (
    df["merchant_state"]
    .map(state_mapping)
    .fillna(df["merchant_state"])
)

# Group locations into broader geographic regions
region_mapping = {
    "Armed Forces Americas": "North America",
    "Alaska": "North America",
    "Alabama": "North America",
    "Arkansas": "North America",
    "Arizona": "North America",
    "California": "North America",
    "Colorado": "North America",
    "Connecticut": "North America",
    "District of Columbia": "North America",
    "Delaware": "North America",
    "Florida": "North America",
    "Georgia": "North America",
    "Hawaii": "North America",
    "Idaho": "North America",
    "Illinois": "North America",
    "Indiana": "North America",
    "Iowa": "North America",
    "Kansas": "North America",
    "Kentucky": "North America",
    "Louisiana": "North America",
    "Maine": "North America",
    "Maryland": "North America",
    "Massachusetts": "North America",
    "Michigan": "North America",
    "Minnesota": "North America",
    "Mississippi": "North America",
    "Missouri": "North America",
    "Montana": "North America",
    "Nebraska": "North America",
    "Nevada": "North America",
    "New Hampshire": "North America",
    "New Jersey": "North America",
    "New Mexico": "North America",
    "New York": "North America",
    "North Carolina": "North America",
    "North Dakota": "North America",
    "Ohio": "North America",
    "Oklahoma": "North America",
    "Oregon": "North America",
    "Pennsylvania": "North America",
    "Rhode Island": "North America",
    "South Carolina": "North America",
    "South Dakota": "North America",
    "Tennessee": "North America",
    "Texas": "North America",
    "Utah": "North America",
    "Vermont": "North America",
    "Virginia": "North America",
    "Washington": "North America",
    "West Virginia": "North America",
    "Wisconsin": "North America",
    "Wyoming": "North America",
    "Canada": "North America",
    "Mexico": "North America",
    "Belize": "North America",
    "Costa Rica": "North America",
    "Dominican Republic": "North America",
    "Haiti": "North America",
    "Honduras": "North America",
    "Jamaica": "North America",
    "Aruba": "North America",
    "The Bahamas": "North America",
    "Trinidad and Tobago": "North America",

    "Argentina": "South America",
    "Brazil": "South America",
    "Colombia": "South America",
    "Peru": "South America",
    "Uruguay": "South America",

    "Andorra": "Europe",
    "Austria": "Europe",
    "Auvergne-Rhône-Alpes": "Europe",
    "Bavaria": "Europe",
    "Belgium": "Europe",
    "Berlin": "Europe",
    "Bosnia and Herzegovina": "Europe",
    "Brussels": "Europe",
    "Campania": "Europe",
    "Capital Region": "Europe",
    "Catalonia": "Europe",
    "Community of Madrid": "Europe",
    "Czech Republic": "Europe",
    "Denmark": "Europe",
    "Finland": "Europe",
    "France": "Europe",
    "Geneva": "Europe",
    "Germany": "Europe",
    "Greece": "Europe",
    "Hamburg": "Europe",
    "Hungary": "Europe",
    "Ireland": "Europe",
    "Italy": "Europe",
    "Lazio": "Europe",
    "Leinster": "Europe",
    "Lisbon": "Europe",
    "Lithuania": "Europe",
    "Lombardy": "Europe",
    "Luxembourg": "Europe",
    "Macedonia": "Europe",
    "Masovian": "Europe",
    "Netherlands": "Europe",
    "North Holland": "Europe",
    "Norway": "Europe",
    "Oslo": "Europe",
    "Poland": "Europe",
    "Porto": "Europe",
    "Portugal": "Europe",
    "Prague": "Europe",
    "Provence-Alpes-Côte d'Azur": "Europe",
    "Romania": "Europe",
    "Russia": "Europe",
    "South Holland": "Europe",
    "Spain": "Europe",
    "Stockholm": "Europe",
    "Sweden": "Europe",
    "Switzerland": "Europe",
    "Uusimaa": "Europe",
    "United Kingdom": "Europe",
    "Valencian Community": "Europe",
    "Vatican City": "Europe",
    "Vienna": "Europe",
    "Zurich": "Europe",
    "Île-de-France": "Europe",

    "China": "Asia",
    "Hong Kong": "Asia",
    "India": "Asia",
    "Indonesia": "Asia",
    "Japan": "Asia",
    "Malaysia": "Asia",
    "Mongolia": "Asia",
    "Pakistan": "Asia",
    "Philippines": "Asia",
    "Singapore": "Asia",
    "South Korea": "Asia",
    "Sri Lanka": "Asia",
    "Taiwan": "Asia",
    "Thailand": "Asia",

    "Iran": "Middle East",
    "Saudi Arabia": "Middle East",
    "Turkey": "Middle East",
    "United Arab Emirates": "Middle East",

    "Burkina Faso": "Africa",
    "Cabo Verde": "Africa",
    "Cote d'Ivoire": "Africa",
    "Egypt": "Africa",
    "Equatorial Guinea": "Africa",
    "Kenya": "Africa",
    "Nigeria": "Africa",
    "South Africa": "Africa",
    "South Sudan": "Africa",

    "Australia": "Oceania",
    "New Zealand": "Oceania",
    "Papua New Guinea": "Oceania"
}

# Create the regional feature
df["merchant_region"] = (
    df["merchant_state_normalized"]
    .map(region_mapping)
    .fillna("Unknown")
)

# Remove intermediate columns
df = df.drop(
    columns=["merchant_state", "merchant_state_normalized"]
)

In [234]:
from sklearn.preprocessing import OneHotEncoder

# Categorical features to encode
categorical_features = [
    "card_usage_method",
    "card_brand",
    "card_type",
    "merchant_region"
]

# Initialize the encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Encode categorical features
encoded_data = encoder.fit_transform(
    df[categorical_features]
)

# Create encoded columns
encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_features),
    index=df.index
)

# Replace categorical columns with encoded columns
df = pd.concat(
    [
        df.drop(columns=categorical_features),
        encoded_df
    ],
    axis=1
)

In [235]:
df.dtypes

amount                           float64
merchant_category_code             int64
transaction_error                  int64
age                                int64
gender                             int64
annual_income                    float64
credit_score                       int64
number_of_cards                    int64
has_chip                           int64
credit_limit                     float64
card_usage_method_Chip           float64
card_usage_method_Contactless    float64
card_usage_method_Online         float64
card_usage_method_Swipe          float64
card_brand_Amex                  float64
card_brand_Discover              float64
card_brand_Mastercard            float64
card_brand_Visa                  float64
card_type_Credit                 float64
card_type_Debit                  float64
card_type_Debit (Prepaid)        float64
merchant_region_Africa           float64
merchant_region_Asia             float64
merchant_region_Europe           float64
merchant_region_

In [236]:
# Group MCC codes into broader merchant category families
def map_mcc_family(mcc):
    if 3000 <= mcc < 4000:
        return "Travel_Transport"
    elif 4000 <= mcc < 5000:
        return "Transport_Utilities"
    elif 5000 <= mcc < 6000:
        return "Retail"
    elif 6000 <= mcc < 7000:
        return "Financial"
    elif 7000 <= mcc < 8000:
        return "Services_Leisure"
    elif 8000 <= mcc < 9000:
        return "Professional_Medical"
    elif 9000 <= mcc < 10000:
        return "Government"
    else:
        return "Other"


# Replace MCC codes with their corresponding family
df["merchant_category_code"] = (
    df["merchant_category_code"].apply(map_mcc_family)
)

In [237]:
# Categorical feature to encode
categorical_features = [
    "merchant_category_code"
]

# Initialize the encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Encode merchant category families
encoded_data = encoder.fit_transform(
    df[categorical_features]
)

# Create encoded columns
encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_features),
    index=df.index
)

# Replace the categorical column with encoded columns
df = pd.concat(
    [
        df.drop(columns=categorical_features),
        encoded_df
    ],
    axis=1
)

In [238]:
import numpy as np

# Apply signed log transformation to reduce amount skewness
df["amount"] = (
    np.sign(df["amount"])
    * np.log1p(np.abs(df["amount"]))
)

In [239]:
# Remove transactions with invalid annual income values
df = df[df["annual_income"] != 1].copy()

## 3. Last check

In [240]:
df.shape

(201418, 36)

In [241]:
df.dtypes

amount                                         float64
transaction_error                                int64
age                                              int64
gender                                           int64
annual_income                                  float64
credit_score                                     int64
number_of_cards                                  int64
has_chip                                         int64
credit_limit                                   float64
card_usage_method_Chip                         float64
card_usage_method_Contactless                  float64
card_usage_method_Online                       float64
card_usage_method_Swipe                        float64
card_brand_Amex                                float64
card_brand_Discover                            float64
card_brand_Mastercard                          float64
card_brand_Visa                                float64
card_type_Credit                               float64
card_type_

In [242]:
print(df.isna().sum().sum())

0


In [243]:
# Save the preprocessed training dataset
df.to_csv("dataset/anomaly_detection_training_dataset.csv", index=False)